## Return Early, Return Often: Guard Clauses & Early Returns

## The Problem with Deep Nesting

When a function checks multiple conditions before doing its real work, the code drifts rightward into a "pyramid of doom":

```python
# Nested — the happy path is buried on line 8
def process_payment(user, cart, payment):
    if user is not None:
        if user.is_active:
            if len(cart) > 0:
                if payment.is_valid():
                    charge(user, cart, payment)
                    return "success"
                else:
                    return "invalid payment"
            else:
                return "empty cart"
        else:
            return "inactive user"
    else:
        return "no user"
```

A **guard clause** inverts each condition and returns (or raises) immediately, keeping the main logic flat:

```python
# Guard clauses — preconditions up front, happy path clear at the bottom
def process_payment(user, cart, payment):
    if user is None:          return "no user"
    if not user.is_active:    return "inactive user"
    if len(cart) == 0:        return "empty cart"
    if not payment.is_valid(): return "invalid payment"

    charge(user, cart, payment)
    return "success"
```

### Learning Goals
- Identify unnecessary nesting caused by positive-condition if/else chains
- Apply guard clauses to handle error / edge cases up front
- Use early `return`, `raise`, and `continue` to flatten loops
- Understand the rule: **check what's wrong first, then do the work**

## Quick Reference

### The Guard Clause Pattern
```python
# Step 1: list all the reasons you CANNOT proceed
# Step 2: return / raise for each one at the top
# Step 3: write the happy path with no extra indentation

def do_thing(x, y):
    if x is None:  raise ValueError("x is required")
    if y <= 0:     raise ValueError("y must be positive")
    # Happy path — no extra nesting needed
    return x / y
```

### Early `return` in a loop
```python
# ❌ Flag variable + full loop
def has_admin(users):
    found = False
    for u in users:
        if u.role == "admin":
            found = True
    return found

# ✅ Return as soon as you know
def has_admin(users):
    for u in users:
        if u.role == "admin":
            return True
    return False
```

### `continue` to skip rather than nest
```python
# ❌ Nested
for item in items:
    if item.is_active:
        process(item)

# ✅ Guard with continue
for item in items:
    if not item.is_active:
        continue
    process(item)
```

## Warm-up — Count the Nesting Levels

For each snippet, answer in the markdown cell below:
1. What is the **maximum indentation depth** of the happy path?
2. What are all the **exit conditions** (reasons the function returns early or differently)?
3. Which conditions could become guard clauses?

In [ ]:
# Snippet A
def send_email(user, message):
    if user is not None:
        if user.email:
            if len(message) > 0:
                deliver(user.email, message)
                return True
            else:
                return False
        else:
            return False
    else:
        return False

**Snippet A analysis** (double-click to edit):

1. Max depth of happy path: 
2. Exit conditions: 
3. Which should be guard clauses: 

In [ ]:
# Snippet B
def find_discount(user_type, amount):
    if amount > 0:
        if user_type == "member":
            if amount > 100:
                return amount * 0.20
            else:
                return amount * 0.10
        else:
            if amount > 100:
                return amount * 0.05
            else:
                return 0
    else:
        return 0

**Snippet B analysis**:

1. Max depth of happy path: 
2. Exit conditions: 
3. Which should be guard clauses: 

## Exercise 1 — Simple Guard Clauses

Rewrite each function using guard clauses to eliminate unnecessary nesting.  
Output must remain identical.

In [ ]:
# Original 1a
def divide(a, b):
    if b != 0:
        result = a / b
        return result
    else:
        return None

print(divide(10, 2))   # 5.0
print(divide(10, 0))   # None

In [ ]:
# Your version of 1a
def divide(a: float, b: float) -> float | None:
    # YOUR CODE HERE
    pass

print(divide(10, 2))   # 5.0
print(divide(10, 0))   # None

In [ ]:
# Original 1b
def get_user_greeting(user):
    if user is not None:
        if user.get("name"):
            if user.get("is_active"):
                return f"Welcome back, {user['name']}!"
            else:
                return "Your account is inactive."
        else:
            return "Hello, stranger!"
    else:
        return "No user provided."

print(get_user_greeting(None))
print(get_user_greeting({}))
print(get_user_greeting({"name": "Alice", "is_active": False}))
print(get_user_greeting({"name": "Alice", "is_active": True}))

In [ ]:
# Your version of 1b — all guards at the top, happy path last
def get_user_greeting(user: dict | None) -> str:
    # YOUR CODE HERE
    pass

print(get_user_greeting(None))                                   # No user provided.
print(get_user_greeting({}))                                     # Hello, stranger!
print(get_user_greeting({"name": "Alice", "is_active": False}))  # Your account is inactive.
print(get_user_greeting({"name": "Alice", "is_active": True}))   # Welcome back, Alice!

In [1]:
# Original 1c — file processing with multiple preconditions
import os

def read_config(filepath):
    if filepath is not None:
        if isinstance(filepath, str):
            if filepath.endswith(".json"):
                if os.path.exists(filepath):
                    with open(filepath) as f:
                        return f.read()
                else:
                    raise FileNotFoundError(f"{filepath} does not exist")
            else:
                raise ValueError("Only .json files are supported")
        else:
            raise TypeError("filepath must be a string")
    else:
        raise ValueError("filepath cannot be None")

In [ ]:
# Your version of 1c
import os

def read_config(filepath: str) -> str:
    # YOUR CODE HERE
    pass

## Exercise 2 — Early Return in Loops

Replace boolean flag patterns and unnecessary full-loop scans with early `return` or `continue`.

In [ ]:
# Original 2a — flag variable pattern
def contains_negative(numbers):
    found = False
    for n in numbers:
        if n < 0:
            found = True
    return found

print(contains_negative([1, 2, -3, 4]))  # True
print(contains_negative([1, 2, 3, 4]))   # False

In [ ]:
# Your version of 2a — return as soon as you know the answer
def contains_negative(numbers: list[float]) -> bool:
    # YOUR CODE HERE
    pass

print(contains_negative([1, 2, -3, 4]))  # True
print(contains_negative([1, 2, 3, 4]))   # False

In [ ]:
# Original 2b — deep nest inside a loop
def process_orders(orders):
    results = []
    for order in orders:
        if order.get("status") == "pending":
            if order.get("amount", 0) > 0:
                if order.get("customer_id"):
                    results.append(f"Processing order {order['id']} for {order['customer_id']}")
    return results

orders = [
    {"id": 1, "status": "pending",   "amount": 50,  "customer_id": "C1"},
    {"id": 2, "status": "shipped",   "amount": 20,  "customer_id": "C2"},
    {"id": 3, "status": "pending",   "amount": 0,   "customer_id": "C3"},
    {"id": 4, "status": "pending",   "amount": 100, "customer_id": None},
    {"id": 5, "status": "pending",   "amount": 75,  "customer_id": "C5"},
]
print(process_orders(orders))

In [ ]:
# Your version of 2b — use `continue` as a guard inside the loop
def process_orders(orders: list[dict]) -> list[str]:
    results = []
    for order in orders:
        # Guard clauses with `continue` go here
        # YOUR CODE HERE
        pass
    return results

orders = [
    {"id": 1, "status": "pending",   "amount": 50,  "customer_id": "C1"},
    {"id": 2, "status": "shipped",   "amount": 20,  "customer_id": "C2"},
    {"id": 3, "status": "pending",   "amount": 0,   "customer_id": "C3"},
    {"id": 4, "status": "pending",   "amount": 100, "customer_id": None},
    {"id": 5, "status": "pending",   "amount": 75,  "customer_id": "C5"},
]
print(process_orders(orders))  # orders 1 and 5 only

## Exercise 3 — Raising Instead of Returning

Sometimes the right guard clause is `raise`, not `return`.  
Refactor each function so invalid inputs are caught immediately with a meaningful exception.

In [ ]:
                                                       # Original 3 — delayed or silent failure
def calculate_bmi(weight_kg, height_m):
    if weight_kg > 0:
        if height_m > 0:
            bmi = weight_kg / (height_m ** 2)
            if bmi < 18.5:
                return bmi, "Underweight"
            elif bmi < 25:
                return bmi, "Normal"
            elif bmi < 30:
                return bmi, "Overweight"
            else:
                return bmi, "Obese"
        else:
            return None, "Invalid height"
    else:
        return None, "Invalid weight"

print(calculate_bmi(70, 1.75))
print(calculate_bmi(-10, 1.75))
print(calculate_bmi(70, 0))

In [ ]:
# Your version of Exercise 3
# Guard invalid inputs by raising ValueError with a clear message.
# The return type becomes tuple[float, str] — always valid when returned.

def calculate_bmi(weight_kg: float, height_m: float) -> tuple[float, str]:
    # YOUR GUARD CLAUSES HERE (raise ValueError for bad inputs)

    # Happy path — runs only with valid input
    # YOUR CODE HERE
    pass

print(calculate_bmi(70, 1.75))  # (~22.9, 'Normal')
try:
    calculate_bmi(-10, 1.75)
except ValueError as e:
    print(e)   # weight_kg must be positive
try:
    calculate_bmi(70, 0)
except ValueError as e:
    print(e)   # height_m must be positive

## Hints

**General rule:** Invert every `if condition: do_work` into `if NOT condition: return early`. Then the work sits at the bottom with no extra indentation.

**Exercise 1b:** You have four guard conditions. Write four one-line guards, then one `return` for the happy path.

**Exercise 1c:** Each `else: raise` becomes the guard. Flip the condition and raise at the top.

**Exercise 2a:** As soon as you find a negative number you know the answer — no need to keep looping. Use `return True` inside the loop and `return False` after.

**Exercise 2b:** Each nested `if` inside the loop becomes `if not condition: continue`.

**Exercise 3:** Returning `None, "error message"` forces the caller to check the second element every time. Raising `ValueError` immediately is cleaner and harder to accidentally ignore.

## Model Solutions

In [ ]:
print("=== Exercise 1a ===")
def divide(a: float, b: float) -> float | None:
    if b == 0:
        return None
    return a / b

print(divide(10, 2))   # 5.0
print(divide(10, 0))   # None

In [ ]:
print("=== Exercise 1b ===")
def get_user_greeting(user: dict | None) -> str:
    if user is None:           return "No user provided."
    if not user.get("name"):   return "Hello, stranger!"
    if not user.get("is_active"): return "Your account is inactive."
    return f"Welcome back, {user['name']}!"

print(get_user_greeting(None))
print(get_user_greeting({}))
print(get_user_greeting({"name": "Alice", "is_active": False}))
print(get_user_greeting({"name": "Alice", "is_active": True}))

In [ ]:
print("=== Exercise 1c ===")
import os

def read_config(filepath: str) -> str:
    if filepath is None:              raise ValueError("filepath cannot be None")
    if not isinstance(filepath, str): raise TypeError("filepath must be a string")
    if not filepath.endswith(".json"): raise ValueError("Only .json files are supported")
    if not os.path.exists(filepath):  raise FileNotFoundError(f"{filepath} does not exist")
    with open(filepath) as f:
        return f.read()

try:
    read_config(None)
except ValueError as e:
    print(e)

In [ ]:
print("=== Exercise 2a ===")
def contains_negative(numbers: list[float]) -> bool:
    for n in numbers:
        if n < 0:
            return True
    return False

print(contains_negative([1, 2, -3, 4]))  # True
print(contains_negative([1, 2, 3, 4]))   # False

In [ ]:
print("=== Exercise 2b ===")
def process_orders(orders: list[dict]) -> list[str]:
    results = []
    for order in orders:
        if order.get("status") != "pending":  continue
        if order.get("amount", 0) <= 0:       continue
        if not order.get("customer_id"):      continue
        results.append(f"Processing order {order['id']} for {order['customer_id']}")
    return results

orders = [
    {"id": 1, "status": "pending",   "amount": 50,  "customer_id": "C1"},
    {"id": 2, "status": "shipped",   "amount": 20,  "customer_id": "C2"},
    {"id": 3, "status": "pending",   "amount": 0,   "customer_id": "C3"},
    {"id": 4, "status": "pending",   "amount": 100, "customer_id": None},
    {"id": 5, "status": "pending",   "amount": 75,  "customer_id": "C5"},
]
print(process_orders(orders))

In [ ]:
print("=== Exercise 3 ===")
def calculate_bmi(weight_kg: float, height_m: float) -> tuple[float, str]:
    if weight_kg <= 0: raise ValueError("weight_kg must be positive")
    if height_m  <= 0: raise ValueError("height_m must be positive")

    bmi = weight_kg / (height_m ** 2)
    if bmi < 18.5:  category = "Underweight"
    elif bmi < 25:  category = "Normal"
    elif bmi < 30:  category = "Overweight"
    else:           category = "Obese"
    return round(bmi, 1), category

print(calculate_bmi(70, 1.75))
try:
    calculate_bmi(-10, 1.75)
except ValueError as e:
    print(e)
try:
    calculate_bmi(70, 0)
except ValueError as e:
    print(e)

## Bonus — Flatten the Pyramid

The function below has **four levels** of nesting and uses flag variables throughout.  
Refactor it to use guard clauses and early returns so the happy path has **no extra indentation**.

You should be able to reduce the body to around 10 lines.

In [ ]:
# --- ORIGINAL ---
def register_user(username, email, age, invite_code):
    success = False
    message = ""
    if username:
        if len(username) >= 3:
            if email and "@" in email:
                if age is not None:
                    if age >= 18:
                        if invite_code == "WELCOME2024":
                            success = True
                            message = f"User {username} registered successfully."
                        else:
                            message = "Invalid invite code."
                    else:
                        message = "Must be 18 or older."
                else:
                    message = "Age is required."
            else:
                message = "Invalid email address."
        else:
            message = "Username must be at least 3 characters."
    else:
        message = "Username is required."
    return success, message

print(register_user("Al", "al@x.com", 20, "WELCOME2024"))
print(register_user("Alice", "alicex.com", 20, "WELCOME2024"))
print(register_user("Alice", "al@x.com", 20, "WELCOME2024"))

In [ ]:
# --- YOUR CLEAN VERSION ---
VALID_INVITE_CODE   = "WELCOME2024"
MIN_USERNAME_LENGTH = 3
MIN_REGISTRATION_AGE = 18

def register_user(username: str, email: str, age: int | None, invite_code: str) -> tuple[bool, str]:
    # YOUR GUARD CLAUSES AND HAPPY PATH HERE
    pass

print(register_user("Al", "al@x.com", 20, "WELCOME2024"))     # (False, 'Username must be...')
print(register_user("Alice", "alicex.com", 20, "WELCOME2024")) # (False, 'Invalid email...')
print(register_user("Alice", "al@x.com", 20, "WELCOME2024"))   # (True,  'User Alice registered...')